# compute launch peaks physical

For each rocket launch window in an input CSV, read waveforms from an SDS archive,
preprocess (incl. response removal) using flovopy, compute amp/energy/fft metrics
via EnhancedStream.ampengfft(), and save per-launch outputs:
```
  - <outdir>/<slug>_<start>.mseed
  - <outdir>/<slug>_<start>.csv           (per-trace metrics; flattened)
  - <outdir>/<slug>_<start>_station.csv   (station-level metrics; if available)
  - optional: <outdir>/<slug>_<start>.pkl
```
Also writes an index CSV summarizing which launches were processed and where outputs live.

Assumptions:
- Input CSV has at least: window_start, window_end
- Optional metadata columns: slug, name, launch_designator, SLC, success

This script:
- Reads ONCE per launch window (padded), then filters traces by network/station/channel patterns.
- Runs preprocess_stream on padded data.
- Trims to the exact launch window AFTER preprocessing to avoid taper bias.
- Recomputes ampengfft on the trimmed window so metrics reflect the launch window only.


In [1]:
from __future__ import annotations

import argparse
import fnmatch
import os
import re
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from obspy import UTCDateTime, read_inventory
from obspy.core.util.attribdict import AttribDict

from flovopy.sds.sds import SDSobj
from flovopy.enhanced.stream import EnhancedStream
from flovopy.core.preprocess import preprocess_stream
from flovopy.processing.spectrograms import icewebSpectrogram


In [2]:


# -----------------------------
# Logging
# -----------------------------
def log(msg: str, verbose: int = 0, level: int = 1) -> None:
    if verbose >= level:
        print(msg, flush=True)


# -----------------------------
# Utils
# -----------------------------
def parse_utc(s: Optional[str]) -> Optional[pd.Timestamp]:
    if not s:
        return None
    return pd.to_datetime(s, utc=True, errors="coerce")


def window_overlaps(a0: pd.Timestamp, a1: pd.Timestamp, b0: pd.Timestamp, b1: pd.Timestamp) -> bool:
    return (a0 <= b1) and (a1 >= b0)


def match_any(value: str, patterns: List[str]) -> bool:
    if not patterns:
        return True
    return any(fnmatch.fnmatch(value, pat) for pat in patterns)


def is_pressure_channel(chan: str) -> bool:
    # Pressure/infrasound: ?D? (second character is 'D')
    return bool(chan) and len(chan) >= 2 and chan[1].upper() == "D"


def safe_slug(s: str) -> str:
    s = (s or "").strip()
    if not s:
        return "launch"
    s = s.lower()
    s = re.sub(r"[^a-z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s or "launch"


# -----------------------------
# Flattening for metrics export
# -----------------------------
def _flatten(obj: Any, prefix: str = "", out: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """
    Recursively flatten dict-like objects (including ObsPy AttribDict) into key/value pairs.

    Examples:
      {"a": {"b": 1}} -> {"a_b": 1}
      AttribDict({"sam": AttribDict({"values": AttribDict({"low": 1})})})
        -> {"sam_values_low": 1}

    Lists/tuples/np arrays are left as-is (stored in CSV as a stringified object by pandas).
    """
    if out is None:
        out = {}

    if obj is None:
        return out

    # Dict-like (AttribDict has .items but isn't a dict subclass)
    if hasattr(obj, "items"):
        for k, v in obj.items():
            key = f"{prefix}{k}" if not prefix else f"{prefix}_{k}"
            if hasattr(v, "items"):
                _flatten(v, key, out)
            else:
                out[key] = v
        return out

    # Non-dict-like
    if prefix:
        out[prefix] = obj
    return out


# -----------------------------
# Save helpers (without requiring you to edit flovopy code right now)
# -----------------------------
def save_enhancedstream_bundle(
    est: EnhancedStream,
    basepath: str,
    *,
    save_pickle: bool = False,
    verbose: int = 0,
) -> Dict[str, str]:
    """
    Save:
      - waveform: basepath.mseed
      - per-trace metrics: basepath.csv     (flattened metrics)
      - station metrics: basepath_station.csv (if available)
      - optional pickle: basepath.pkl

    Returns dict of written paths.
    """
    import pickle

    # strip .mseed if user passed it
    if basepath.endswith(".mseed"):
        basepath = basepath[:-6]

    outdir = os.path.dirname(basepath)
    if outdir and not os.path.exists(outdir):
        os.makedirs(outdir, exist_ok=True)

    written: Dict[str, str] = {}

    # 1) waveform
    mseed_path = basepath + ".mseed"
    est.write(mseed_path, format="MSEED")
    written["mseed"] = mseed_path

    # 2) per-trace metrics
    trace_rows: List[Dict[str, Any]] = []
    for tr in est:
        s = tr.stats
        row: Dict[str, Any] = {
            "id": tr.id,
            "network": getattr(s, "network", ""),
            "station": getattr(s, "station", ""),
            "location": getattr(s, "location", ""),
            "channel": getattr(s, "channel", ""),
            "starttime": s.starttime,
            "endtime": s.endtime,
            "Fs": getattr(s, "sampling_rate", None),
            "calib": getattr(s, "calib", None),
            "units": getattr(s, "units", None),
            "quality": getattr(s, "quality_factor", None),
            "is_pressure": is_pressure_channel(getattr(s, "channel", "")),
        }

        # spectrum (if present)
        if hasattr(s, "spectrum"):
            row.update(_flatten(s.spectrum, prefix="spectrum"))

        # metrics (flatten deeply, including AttribDict)
        if hasattr(s, "metrics"):
            row.update(_flatten(s.metrics, prefix=""))

        # coordinates (if present)
        if hasattr(s, "coordinates"):
            try:
                row["latitude"] = s.coordinates.latitude
                row["longitude"] = s.coordinates.longitude
                row["elevation"] = s.coordinates.elevation
            except Exception:
                pass

        trace_rows.append(row)

    df_traces = pd.DataFrame(trace_rows)
    csv_path = basepath + ".csv"
    df_traces.to_csv(csv_path, index=False)
    written["trace_csv"] = csv_path

    # 3) station-level metrics (if EnhancedStream provides them)
    station_csv = basepath + "_station.csv"
    wrote_station = False
    try:
        sdf = getattr(est, "station_metrics", None)
        if sdf is None or getattr(sdf, "empty", True):
            # Some flovopy versions compute station metrics lazily
            if hasattr(est, "_station_level_metrics"):
                sdf = est._station_level_metrics()
        if sdf is not None and not getattr(sdf, "empty", True):
            sdf.to_csv(station_csv, index=False)
            wrote_station = True
            written["station_csv"] = station_csv
    except Exception as e:
        log(f"[WARN] station-level CSV not written: {e}", verbose, 1)

    # 4) pickle (optional)
    if save_pickle:
        pkl_path = basepath + ".pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump(est, f)
        written["pickle"] = pkl_path

    log(f"[✓] Saved {len(est)} traces to {mseed_path} and metrics to {csv_path}", verbose, 1)
    if wrote_station:
        log(f"[✓] Station metrics: {station_csv}", verbose, 2)
    return written



In [3]:
events_csv = 'all_florida_launches_with_seed_ids.csv'
stationxml = '/Users/glennthompson/Dropbox/KSC_RocketSeis_responses_computed.xml'
sds_root = '/Volumes/data/remastered/SDS_KSC'
date_from = "2016-02-20"
date_to = "2022-12-05"
outdir = "event_outputs"
index_out = "rocket_events_recorded.csv"
pad_s = 60
verbose = 2

In [4]:
date_from = parse_utc(date_from)
date_to = parse_utc(date_to)
if (date_from is not None) and (date_to is None):
    date_to = pd.Timestamp.max.tz_localize("UTC")
if (date_to is not None) and (date_from is None):
    date_from = pd.Timestamp.min.tz_localize("UTC")


log(f"Loading CSV: {events_csv}", verbose, 1)
df = pd.read_csv(events_csv)
if "window_start" not in df.columns or "window_end" not in df.columns:
    raise ValueError("events CSV must contain window_start and window_end columns")

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

# Filter by date overlap if requested
if date_from is not None and date_to is not None:
    before = len(df)
    mask = []
    for r in df.itertuples(index=False):
        if pd.isna(r.window_start) or pd.isna(r.window_end):
            mask.append(False)
        else:
            mask.append(window_overlaps(r.window_start, r.window_end, date_from, date_to))
    df = df.loc[mask].copy()
    log(f"Date filter [{date_from} .. {date_to}] kept {len(df)}/{before} launches", verbose, 1)

log(f"Loading StationXML: {stationxml}", verbose, 1)
inv = read_inventory(stationxml)

log(f"Opening SDS via SDSobj: {sds_root}", verbose, 1)
sdsobject = SDSobj(sds_root)

os.makedirs(outdir, exist_ok=True)

index_rows: List[Dict[str, Any]] = []
launch_count = 0


Loading CSV: all_florida_launches_with_seed_ids.csv
Date filter [2016-02-20 00:00:00+00:00 .. 2022-12-05 00:00:00+00:00] kept 189/408 launches
Loading StationXML: /Users/glennthompson/Dropbox/KSC_RocketSeis_responses_computed.xml
Opening SDS via SDSobj: /Volumes/data/remastered/SDS_KSC


In [5]:
print(df.columns)
print(df.head())

Index(['name', 'slug', 'launch_designator', 'SLC', 'success', 'net',
       'window_start', 'window_end', 'SEED_ids'],
      dtype='object')
                                                name  \
1                       Falcon 9 Full Thrust | SES-9   
2  Atlas V 401 | Cygnus CRS OA-6 (S.S. Rick Husband)   
3                   Falcon 9 Full Thrust | SpX CRS-8   
4                    Falcon 9 Full Thrust | JCSAT-14   
5                   Falcon 9 Full Thrust | Thaicom 8   

                                          slug launch_designator  \
1                   falcon-9-full-thrust-ses-9          2016-013   
2  atlas-v-401-cygnus-crs-oa-6-ss-rick-husband          2016-019   
3               falcon-9-full-thrust-spx-crs-8          2016-024   
4                falcon-9-full-thrust-jcsat-14          2016-028   
5               falcon-9-full-thrust-thaicom-8          2016-031   

                       SLC  success                   net  \
1  Space Launch Complex 40     True  2016-03-04T23:3

# Load the raw Stream for each rocket event, and save to MiniSEED file

In [6]:
def bad_row_info(log_message: str, r: pd.Series, index_rows: list, slug: str, basepath: str, e: Exception, verbose_level: int ) -> str:
    log(f"  {log_message} {type(e).__name__}: {e}", verbose=verbose, level=verbose_level)
    index_rows.append(dict(
        slug=slug,
        name=getattr(r, "name", ""),
        window_start=str(r.window_start),
        window_end=str(r.window_end),
        status="read_fail",
        error=f"{type(e).__name__}: {e}",
        basepath=basepath,
    ))


In [7]:

from flovopy.core.miniseed_io import write_mseed

if os.path.exists(index_out):
    event_df = pd.read_csv(index_out)
else:
    
    event_lod = []
    for r in df.itertuples(index=False):
        if pd.isna(r.window_start) or pd.isna(r.window_end):
            continue

        launch_count += 1

        t0 = UTCDateTime(r.window_start.to_pydatetime())
        t1 = UTCDateTime(r.window_end.to_pydatetime())
        tp0 = t0 - float(pad_s)
        tp1 = t1 + float(pad_s)

        slug = safe_slug(getattr(r, "slug", "") or getattr(r, "name", "") or f"launch_{launch_count:04d}")
        start_tag = r.window_start.strftime("%Y%m%dT%H%M%SZ")
        eventdir = os.path.join(outdir, start_tag)
        os.makedirs(eventdir, exist_ok=True)
        #base = f"{start_tag}_{slug}"
        basepath = os.path.join(eventdir, slug)

        log(f"\nLaunch {launch_count}: {slug}  {r.window_start} -> {r.window_end}", verbose, 1)

        # --- Read once per launch window (padded) ---
        try:
            sdsobject.read(tp0, tp1)
            st = sdsobject.stream
        except Exception as e:
            bad_row_info("READ FAIL:", r, index_rows, slug, basepath, e, verbose_level=1)
            continue

        if not st or len(st) == 0:
            bad_row_info("NO DATA (window):", r, index_rows, slug, basepath, Exception("No traces found in window"), verbose_level=2)
            continue

        mseed_path = basepath + "_raw.mseed"
        write_mseed(st, mseed_path)


        event_lod.append({'t0':t0, 't1':t1, 'slug':slug, 'basepath':basepath, 'mseed_path':mseed_path})

    event_df = pd.DataFrame(event_lod)
    event_df.to_csv(index_out, index=False)
    log(f"\nDone! Processed {len(event_df)} launches. Index saved to {index_out}", verbose, 1)
    print(event_df)



Launch 1: falcon-9-full-thrust-ses-9  2016-03-04 23:35:00+00:00 -> 2016-03-05 01:06:00+00:00

Launch 2: atlas-v-401-cygnus-crs-oa-6-ss-rick-husband  2016-03-23 03:05:52+00:00 -> 2016-03-23 03:35:00+00:00

Launch 3: falcon-9-full-thrust-spx-crs-8  2016-04-08 20:43:32+00:00 -> 2016-04-08 20:43:32+00:00

Launch 4: falcon-9-full-thrust-jcsat-14  2016-05-06 05:21:00+00:00 -> 2016-05-06 07:21:00+00:00
  NO DATA (window): Exception: No traces found in window

Launch 5: falcon-9-full-thrust-thaicom-8  2016-05-27 21:39:00+00:00 -> 2016-05-27 23:39:00+00:00

Launch 6: delta-iv-heavy-nrol-37  2016-06-11 17:51:00+00:00 -> 2016-06-11 17:51:00+00:00

Launch 7: falcon-9-full-thrust-eutelsat-117-west-b-abs-2a  2016-06-15 14:29:00+00:00 -> 2016-06-15 15:13:00+00:00

Launch 8: atlas-v-551-muos-5  2016-06-24 14:30:00+00:00 -> 2016-06-24 15:14:00+00:00

Launch 9: falcon-9-full-thrust-spx-crs-9  2016-07-18 04:45:29+00:00 -> 2016-07-18 04:45:29+00:00
  NO DATA (window): Exception: No traces found in window

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/mseed/core.py:826: UserWarning: File will be written with more than one different record lengths.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'record lengths')



Launch 54: atlas-v-551-aehf-4-usa-288  2018-10-17 04:15:00+00:00 -> 2018-10-17 06:15:00+00:00

Launch 55: falcon-9-block-5-eshail-2  2018-11-15 20:46:00+00:00 -> 2018-11-15 22:29:00+00:00

Launch 56: falcon-9-block-5-spx-crs-16  2018-12-05 18:16:16+00:00 -> 2018-12-05 18:16:16+00:00

Launch 57: falcon-9-block-5-gps-iii-sv01  2018-12-23 13:51:00+00:00 -> 2018-12-23 14:17:00+00:00

Launch 58: falcon-9-block-5-nusantara-satu-gto-1-beresheet-s5  2019-02-22 01:45:00+00:00 -> 2019-02-22 02:17:00+00:00

Launch 59: falcon-9-block-5-spx-dm1-demonstration-mission-1  2019-03-02 07:49:00+00:00 -> 2019-03-02 07:49:00+00:00

Launch 60: delta-iv-m54-wgs-10  2019-03-15 22:56:00+00:00 -> 2019-03-16 01:05:00+00:00

Launch 61: falcon-heavy-arabsat-6a  2019-04-11 22:35:00+00:00 -> 2019-04-12 00:31:00+00:00

Launch 62: falcon-9-block-5-spx-crs-17  2019-05-04 06:48:58+00:00 -> 2019-05-04 06:48:58+00:00

Launch 63: falcon-9-block-5-starlink-demo-09  2019-05-24 02:30:00+00:00 -> 2019-05-24 04:00:00+00:00

La

# Let's plot the seismic and infrasound traces separately, and save them to raw PNG files

In [8]:
from flovopy.core.miniseed_io import read_mseed 
for i, row in event_df.iterrows():
    basepath = row['basepath']
    slug = row['slug']
    t0 = row['t0']
    t1 = row['t1']
    print(f"\nProcessing launch {i+1}/{len(event_df)}: {slug}  {t0} -> {t1}")
    try:
        st = read_mseed(row['mseed_path'])
        print(f"  Read {len(st)} traces from {row['mseed_path']}")
      
    except Exception as e:
        print(f"  Failed to read/process {row['mseed_path']}: {type(e).__name__}: {e}")
        continue
    else:
        try:
            st_pp = preprocess_stream(
                st,
                freq=0.4,
                filter_type="highpass",
                inv=None,
                verbose=verbose,
            )
            #st_pp.detrend("linear")
        except Exception as e:
            print(f"  preprocess_stream FAILED: {type(e).__name__}: {e}")
        else:
            st_seismic = st.select(channel="[SBEHCDFG]H*")
            if len(st_seismic) > 0:
                st_seismic.plot(outfile=basepath + "_seismic_raw.png", size=(1200, 800), equal_scale=False)
            st_infrasound = st_pp.select(channel="[SBEHCDFG]D*")
            if len(st_infrasound) > 0:
                st_infrasound.plot(outfile=basepath + "_infrasound_raw.png", size=(1200, 800), equal_scale=False)  
            print(f"  Plotted seismic and infrasound traces for {slug}")




Processing launch 1/140: falcon-9-full-thrust-ses-9  2016-03-04T23:35:00.000000Z -> 2016-03-05T01:06:00.000000Z
  Read 18 traces from event_outputs/20160304T233500Z/falcon-9-full-thrust-ses-9_raw.mseed
  Plotted seismic and infrasound traces for falcon-9-full-thrust-ses-9

Processing launch 2/140: atlas-v-401-cygnus-crs-oa-6-ss-rick-husband  2016-03-23T03:05:52.000000Z -> 2016-03-23T03:35:00.000000Z
  Read 12 traces from event_outputs/20160323T030552Z/atlas-v-401-cygnus-crs-oa-6-ss-rick-husband_raw.mseed
  Plotted seismic and infrasound traces for atlas-v-401-cygnus-crs-oa-6-ss-rick-husband

Processing launch 3/140: falcon-9-full-thrust-spx-crs-8  2016-04-08T20:43:32.000000Z -> 2016-04-08T20:43:32.000000Z
  Read 18 traces from event_outputs/20160408T204332Z/falcon-9-full-thrust-spx-crs-8_raw.mseed
  Plotted seismic and infrasound traces for falcon-9-full-thrust-spx-crs-8

Processing launch 4/140: falcon-9-full-thrust-thaicom-8  2016-05-27T21:39:00.000000Z -> 2016-05-27T23:39:00.000000

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',


  Plotted seismic and infrasound traces for falcon-9-block-4-koreasat-5a

Processing launch 17/140: falcon-9-full-thrust-spx-crs-13  2017-12-15T15:36:09.000000Z -> 2017-12-15T15:36:09.000000Z
  Read 11 traces from event_outputs/20171215T153609Z/falcon-9-full-thrust-spx-crs-13_raw.mseed
  Plotted seismic and infrasound traces for falcon-9-full-thrust-spx-crs-13

Processing launch 18/140: falcon-9-block-4-zuma  2018-01-08T01:00:00.000000Z -> 2018-01-08T03:00:00.000000Z
  Read 5 traces from event_outputs/20180108T010000Z/falcon-9-block-4-zuma_raw.mseed
  Plotted seismic and infrasound traces for falcon-9-block-4-zuma

Processing launch 19/140: atlas-v-411-sbirs-geo-flight-4-sbirs-geo-3  2018-01-20T00:48:00.000000Z -> 2018-01-20T01:32:00.000000Z
  Read 5 traces from event_outputs/20180120T004800Z/atlas-v-411-sbirs-geo-flight-4-sbirs-geo-3_raw.mseed
  Plotted seismic and infrasound traces for atlas-v-411-sbirs-geo-flight-4-sbirs-geo-3

Processing launch 20/140: falcon-9-full-thrust-govsat-s

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_

  Plotted seismic and infrasound traces for falcon-9-full-thrust-govsat-ses-16

Processing launch 21/140: falcon-heavy-demo-test-flight  2018-02-06T18:30:00.000000Z -> 2018-02-06T21:00:00.000000Z
  Read 23 traces from event_outputs/20180206T183000Z/falcon-heavy-demo-test-flight_raw.mseed


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',


  Plotted seismic and infrasound traces for falcon-heavy-demo-test-flight

Processing launch 22/140: atlas-v-541-goes-s  2018-03-01T22:02:00.000000Z -> 2018-03-02T00:02:00.000000Z
  Read 11 traces from event_outputs/20180301T220200Z/atlas-v-541-goes-s_raw.mseed
  Plotted seismic and infrasound traces for atlas-v-541-goes-s

Processing launch 23/140: falcon-9-block-4-hispasat-30w-6-hispasat-1f  2018-03-06T05:33:00.000000Z -> 2018-03-06T07:33:00.000000Z
  Read 11 traces from event_outputs/20180306T053300Z/falcon-9-block-4-hispasat-30w-6-hispasat-1f_raw.mseed
  Plotted seismic and infrasound traces for falcon-9-block-4-hispasat-30w-6-hispasat-1f

Processing launch 24/140: delta-iv-heavy-parker-solar-probe  2018-08-12T07:31:00.000000Z -> 2018-08-12T08:36:00.000000Z
  Read 14 traces from event_outputs/20180812T073100Z/delta-iv-heavy-parker-solar-probe_raw.mseed
  Plotted seismic and infrasound traces for delta-iv-heavy-parker-solar-probe

Processing launch 25/140: falcon-9-block-5-telstar-1

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_

  Plotted seismic and infrasound traces for falcon-9-block-5-nilesat-301

Processing launch 112/140: astra-rocket-3-tropics-1  2022-06-12T16:00:00.000000Z -> 2022-06-12T18:00:00.000000Z
  Read 19 traces from event_outputs/20220612T160000Z/astra-rocket-3-tropics-1_raw.mseed
  Plotted seismic and infrasound traces for astra-rocket-3-tropics-1

Processing launch 113/140: falcon-9-block-5-starlink-group-4-19  2022-06-17T16:08:50.000000Z -> 2022-06-17T16:08:50.000000Z
  Read 19 traces from event_outputs/20220617T160850Z/falcon-9-block-5-starlink-group-4-19_raw.mseed
  Plotted seismic and infrasound traces for falcon-9-block-5-starlink-group-4-19

Processing launch 114/140: falcon-9-block-5-globalstar-2-fm15-usa-328-331  2022-06-19T04:27:36.000000Z -> 2022-06-19T04:27:36.000000Z
  Read 19 traces from event_outputs/20220619T042736Z/falcon-9-block-5-globalstar-2-fm15-usa-328-331_raw.mseed
  Plotted seismic and infrasound traces for falcon-9-block-5-globalstar-2-fm15-usa-328-331

Processing lau

# Let's plot the seismic and infrasound traces separately, and save them to corrected PNG files

In [9]:
from flovopy.core.miniseed_io import read_mseed 

pgv_lod = []
pap_lod = []

for i, row in event_df.iterrows():
    basepath = row['basepath']
    slug = row['slug']
    t0 = row['t0']
    t1 = row['t1']
    print(f"\nProcessing launch {i+1}/{len(event_df)}: {slug}  {t0} -> {t1}")
    try:
        st = read_mseed(row['mseed_path'])
        print(f"  Read {len(st)} traces from {row['mseed_path']}")
      
    except Exception as e:
        print(f"  Failed to read/process {row['mseed_path']}: {type(e).__name__}: {e}")
        continue
    else:
        try:
            st_pp = preprocess_stream(
                st,
                freq=0.4,
                filter_type="highpass",
                inv=inv,
                verbose=verbose,
            )
            #st_pp.detrend("linear")
        except Exception as e:
            print(f"  preprocess_stream FAILED: {type(e).__name__}: {e}")
        else:
            if len(st_pp) == 0:
                print(f"  No traces left after preprocessing for {slug}")
                continue
            st_seismic = st_pp.select(channel="[SBEHCDFG]H*")
            if len(st_seismic) > 0:
                st_seismic.plot(outfile=basepath + "_seismic_corrected.png", size=(1200, 800), equal_scale=False)
                event_dict = {'starttime': t0, 'endtime': t1, 'slug': slug}
                for tr in st_seismic:
                    event_dict[f"{tr.id}"]=np.nanmax(np.abs(tr.data))
                pgv_lod.append(event_dict)
            st_infrasound = st_pp.select(channel="[SBEHCDFG]D*")
            if len(st_infrasound) > 0:
                st_infrasound.plot(outfile=basepath + "_infrasound_corrected.png", size=(1200, 800), equal_scale=False)  
                event_dict = {'starttime': t0, 'endtime': t1, 'slug': slug}
                for tr in st_infrasound:
                    event_dict[f"{tr.id}"]=np.nanmax(np.abs(tr.data))
                pap_lod.append(event_dict)                
            print(f"  Plotted seismic and infrasound traces for {slug}")
pgv_lod_df = pd.DataFrame(pgv_lod)
pap_lod_df = pd.DataFrame(pap_lod)
pgv_lod_df.to_csv("pgv_metrics.csv", index=False)
pap_lod_df.to_csv("pap_metrics.csv", index=False)




Processing launch 1/140: falcon-9-full-thrust-ses-9  2016-03-04T23:35:00.000000Z -> 2016-03-05T01:06:00.000000Z
  Read 18 traces from event_outputs/20160304T233500Z/falcon-9-full-thrust-ses-9_raw.mseed
- removing instrument response
1R.BCHH.00.DD1 units=Pa
- removing instrument response
1R.BCHH.00.DD2 units=Pa
- removing instrument response
1R.BCHH.00.DD3 units=Pa
- removing instrument response
1R.BCHH.00.DHE units=m/s
- removing instrument response
1R.BCHH.00.DHN units=m/s
- removing instrument response
1R.BCHH.00.DHZ units=m/s
- removing instrument response
1R.FIRE.00.DD1 units=Pa
- removing instrument response
1R.FIRE.00.DD2 units=Pa
- removing instrument response
1R.FIRE.00.DD3 units=Pa
- removing instrument response
1R.FIRE.00.DHE units=m/s
- removing instrument response
1R.FIRE.00.DHN units=m/s
- removing instrument response
1R.FIRE.00.DHZ units=m/s
- removing instrument response
1R.TANK.00.DD1 units=Pa
- removing instrument response
1R.TANK.00.DD2 units=Pa
- removing instrument

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',


1R.BCHH2.00.DD4 units=Pa
- removing instrument response
1R.BCHH2.00.DD5 units=Pa
- removing instrument response
1R.BCHH2.00.DD6 units=Pa
- removing instrument response
1R.BCHH2.00.DD7 units=Pa
- removing instrument response
1R.BCHH2.00.DD8 units=Pa
- removing instrument response
1R.BCHH2.00.DD9 units=Pa
  Plotted seismic and infrasound traces for falcon-9-full-thrust-spx-crs-13

Processing launch 18/140: falcon-9-block-4-zuma  2018-01-08T01:00:00.000000Z -> 2018-01-08T03:00:00.000000Z
  Read 5 traces from event_outputs/20180108T010000Z/falcon-9-block-4-zuma_raw.mseed
- removing instrument response
Error removing response for 1R.BCHH1.00.DD1: No matching response information found.
- removing instrument response
1R.BCHH1.00.DD2 units=Pa
- removing instrument response
1R.BCHH1.00.DHE units=m/s
- removing instrument response
1R.BCHH1.00.DHN units=m/s
- removing instrument response
1R.BCHH1.00.DHZ units=m/s
  Plotted seismic and infrasound traces for falcon-9-block-4-zuma

Processing launc

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_

  Read 23 traces from event_outputs/20180206T183000Z/falcon-heavy-demo-test-flight_raw.mseed
- removing instrument response
Error removing response for 1R.BCHH1.00.DD1: No matching response information found.
- removing instrument response
1R.BCHH1.00.DD2 units=Pa
- removing instrument response
1R.BCHH1.00.DHE units=m/s
- removing instrument response
1R.BCHH1.00.DHN units=m/s
- removing instrument response
1R.BCHH1.00.DHZ units=m/s
- removing instrument response
1R.BCHH2.00.DD4 units=Pa
- removing instrument response
1R.BCHH2.00.DD5 units=Pa
- removing instrument response
1R.BCHH2.00.DD6 units=Pa
- removing instrument response
1R.BCHH2.00.DD7 units=Pa
- removing instrument response
1R.BCHH2.00.DD8 units=Pa
- removing instrument response
1R.BCHH2.00.DD9 units=Pa
- removing instrument response
1R.DVEL1.00.DD1 units=Pa
- removing instrument response
1R.DVEL1.00.DD2 units=Pa
- removing instrument response
1R.DVEL1.00.DD3 units=Pa
- removing instrument response
1R.DVEL1.00.DD4 units=Pa
- re

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',


  Plotted seismic and infrasound traces for falcon-heavy-demo-test-flight

Processing launch 22/140: atlas-v-541-goes-s  2018-03-01T22:02:00.000000Z -> 2018-03-02T00:02:00.000000Z
  Read 11 traces from event_outputs/20180301T220200Z/atlas-v-541-goes-s_raw.mseed
- removing instrument response
Error removing response for 1R.BCHH1.00.DD1: No matching response information found.
- removing instrument response
1R.BCHH1.00.DD2 units=Pa
- removing instrument response
1R.BCHH1.00.DHE units=m/s
- removing instrument response
1R.BCHH1.00.DHN units=m/s
- removing instrument response
1R.BCHH1.00.DHZ units=m/s
- removing instrument response
1R.BCHH2.00.DD4 units=Pa
- removing instrument response
1R.BCHH2.00.DD5 units=Pa
- removing instrument response
1R.BCHH2.00.DD6 units=Pa
- removing instrument response
1R.BCHH2.00.DD7 units=Pa
- removing instrument response
1R.BCHH2.00.DD8 units=Pa
- removing instrument response
1R.BCHH2.00.DD9 units=Pa
  Plotted seismic and infrasound traces for atlas-v-541-goe

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_

  Plotted seismic and infrasound traces for falcon-9-block-5-nilesat-301

Processing launch 112/140: astra-rocket-3-tropics-1  2022-06-12T16:00:00.000000Z -> 2022-06-12T18:00:00.000000Z
  Read 19 traces from event_outputs/20220612T160000Z/astra-rocket-3-tropics-1_raw.mseed
- removing instrument response
1R.BCHH2.00.HD4 units=Pa
- removing instrument response
1R.BCHH2.00.HD5 units=Pa
- removing instrument response
1R.BCHH2.00.HD6 units=Pa
- removing instrument response
1R.BCHH2.00.HD7 units=Pa
- removing instrument response
1R.BCHH2.00.HD8 units=Pa
- removing instrument response
1R.BCHH2.00.HD9 units=Pa
- removing instrument response
1R.BCHH4.00.HD2 units=Pa
- removing instrument response
Error removing response for 1R.BCHH4.00.HDF: No matching response information found.
- removing instrument response
1R.BCHH4.00.HHE units=m/s
- removing instrument response
1R.BCHH4.00.HHN units=m/s
- removing instrument response
1R.BCHH4.00.HHZ units=m/s
- removing instrument response
1R.S39A1.00.HDF 

# Try to do this for lots of metrics


In [11]:

import numpy as np
import pandas as pd

from flovopy.core.miniseed_io import read_mseed

# ---- Config ----
SEIS_CLIP = 0.01     # m/s
INFRA_CLIP = 3000.0  # Pa
WIN_SEC = 1.0        # sliding window length for RMS metric

def _safe_float_data(tr):
    """Return float64 data view (safe for abs/percentile) and mask NaNs."""
    x = tr.data.astype(np.float64, copy=False)
    return x

def _clip_series(x, clip_val):
    """Clip x to +/- clip_val. Return clipped array + fraction clipped."""
    if clip_val is None:
        return x, 0.0
    x = np.asarray(x, dtype=np.float64)
    # ignore NaNs in fraction calculation
    finite = np.isfinite(x)
    if not finite.any():
        return x, np.nan
    before = x[finite]
    clipped = np.clip(x, -clip_val, clip_val)
    after = clipped[finite]
    frac_clipped = float(np.mean(np.abs(before) > clip_val))
    return clipped, frac_clipped

def _max_rolling_rms(x, fs, win_sec=1.0):
    """Max RMS in a sliding window using convolution. Robust to spikes if used with clipping."""
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)
    if not finite.any():
        return np.nan
    x = np.nan_to_num(x, nan=0.0)

    nwin = int(round(win_sec * fs))
    nwin = max(1, nwin)

    if nwin >= x.size:
        return float(np.sqrt(np.mean(x**2)))

    x2 = x * x
    kernel = np.ones(nwin, dtype=np.float64) / nwin
    rms_series = np.sqrt(np.convolve(x2, kernel, mode="valid"))
    return float(np.max(rms_series))

def _amp_metrics(tr, clip_val, win_sec=1.0):
    """
    Compute robust amplitude metrics on a trace:
      - peak_abs (after clipping)
      - p99_abs  (after clipping)
      - max_1s_rms (after clipping)
      - spike_ratio = peak / p99  (big values often indicate a spike)
      - frac_clipped
    """
    fs = float(tr.stats.sampling_rate)
    x = _safe_float_data(tr)

    x_clip, frac_clipped = _clip_series(x, clip_val)
    absx = np.abs(x_clip)

    # handle all-NaN edge cases safely
    if not np.isfinite(absx).any():
        return dict(
            peak_abs=np.nan,
            p99_abs=np.nan,
            max_1s_rms=np.nan,
            spike_ratio=np.nan,
            frac_clipped=frac_clipped,
        )

    peak_abs = float(np.nanmax(absx))
    p99_abs = float(np.nanpercentile(absx, 99))
    max_1s_rms = _max_rolling_rms(x_clip, fs, win_sec=win_sec)

    spike_ratio = float(peak_abs / p99_abs) if p99_abs > 0 else np.nan

    return dict(
        peak_abs=peak_abs,
        p99_abs=p99_abs,
        max_1s_rms=max_1s_rms,
        spike_ratio=spike_ratio,
        frac_clipped=frac_clipped,
    )

# ---- Output containers (parallel DataFrames: columns are tr.id) ----
pgv_peak_lod, pgv_p99_lod, pgv_rms1s_lod, pgv_spike_lod, pgv_clipfrac_lod = ([] for _ in range(5))
pap_peak_lod, pap_p99_lod, pap_rms1s_lod, pap_spike_lod, pap_clipfrac_lod = ([] for _ in range(5))

for i, row in event_df.iterrows():
    basepath = row["basepath"]
    slug = row["slug"]
    t0 = row["t0"]
    t1 = row["t1"]
    print(f"\nProcessing launch {i+1}/{len(event_df)}: {slug}  {t0} -> {t1}")

    # --- Read ---
    try:
        st = read_mseed(row["mseed_path"])
        print(f"  Read {len(st)} traces from {row['mseed_path']}")
    except Exception as e:
        print(f"  Failed to read/process {row['mseed_path']}: {type(e).__name__}: {e}")
        continue

    # --- Preprocess ---
    try:
        st_pp = preprocess_stream(
            st,
            freq=0.4,
            filter_type="highpass",
            inv=inv,
            verbose=verbose,
        )
    except Exception as e:
        print(f"  preprocess_stream FAILED: {type(e).__name__}: {e}")
        continue

    if len(st_pp) == 0:
        print(f"  No traces left after preprocessing for {slug}")
        continue

    # --- Trim to event window (metrics must be event-specific) ---
    try:
        st_pp.trim(t0, t1, pad=False)
    except Exception as e:
        print(f"  trim FAILED for {slug}: {type(e).__name__}: {e}")
        continue

    if len(st_pp) == 0 or all(tr.stats.npts < 2 for tr in st_pp):
        print(f"  No usable samples after trimming for {slug}")
        continue

    # --- Channel selection ---
    st_seismic = st_pp.select(channel="[SBEHCDFG]H*")
    st_infrasound = st_pp.select(channel="[SBEHCDFG]D*")

    # --- Optional quick plots (trimmed, corrected) ---
    try:
        if len(st_seismic) > 0:
            st_seismic.plot(outfile=basepath + "_seismic_corrected.png",
                            size=(1200, 800), equal_scale=False)
        if len(st_infrasound) > 0:
            st_infrasound.plot(outfile=basepath + "_infrasound_corrected.png",
                               size=(1200, 800), equal_scale=False)
        print(f"  Plotted seismic and infrasound traces for {slug}")
    except Exception as e:
        print(f"  Plotting WARNING for {slug}: {type(e).__name__}: {e}")

    # ---- Compute metrics: Seismic (m/s) ----
    if len(st_seismic) > 0:
        base = {"starttime": t0, "endtime": t1, "slug": slug}
        d_peak = dict(base)
        d_p99 = dict(base)
        d_rms = dict(base)
        d_spk = dict(base)
        d_cfr = dict(base)

        for tr in st_seismic:
            m = _amp_metrics(tr, clip_val=SEIS_CLIP, win_sec=WIN_SEC)
            d_peak[tr.id] = m["peak_abs"]
            d_p99[tr.id] = m["p99_abs"]
            d_rms[tr.id]  = m["max_1s_rms"]
            d_spk[tr.id]  = m["spike_ratio"]
            d_cfr[tr.id]  = m["frac_clipped"]

        pgv_peak_lod.append(d_peak)
        pgv_p99_lod.append(d_p99)
        pgv_rms1s_lod.append(d_rms)
        pgv_spike_lod.append(d_spk)
        pgv_clipfrac_lod.append(d_cfr)

    # ---- Compute metrics: Infrasound (Pa) ----
    if len(st_infrasound) > 0:
        base = {"starttime": t0, "endtime": t1, "slug": slug}
        d_peak = dict(base)
        d_p99 = dict(base)
        d_rms = dict(base)
        d_spk = dict(base)
        d_cfr = dict(base)

        for tr in st_infrasound:
            m = _amp_metrics(tr, clip_val=INFRA_CLIP, win_sec=WIN_SEC)
            d_peak[tr.id] = m["peak_abs"]
            d_p99[tr.id] = m["p99_abs"]
            d_rms[tr.id]  = m["max_1s_rms"]
            d_spk[tr.id]  = m["spike_ratio"]
            d_cfr[tr.id]  = m["frac_clipped"]

        pap_peak_lod.append(d_peak)
        pap_p99_lod.append(d_p99)
        pap_rms1s_lod.append(d_rms)
        pap_spike_lod.append(d_spk)
        pap_clipfrac_lod.append(d_cfr)

# ---- Save outputs (parallel CSVs) ----
pd.DataFrame(pgv_peak_lod).to_csv("pgv_peak_abs_clipped.csv", index=False)
pd.DataFrame(pgv_p99_lod).to_csv("pgv_p99_abs_clipped.csv", index=False)
pd.DataFrame(pgv_rms1s_lod).to_csv("pgv_max_1s_rms_clipped.csv", index=False)
pd.DataFrame(pgv_spike_lod).to_csv("pgv_spike_ratio_peak_over_p99.csv", index=False)
pd.DataFrame(pgv_clipfrac_lod).to_csv("pgv_fraction_samples_clipped.csv", index=False)

pd.DataFrame(pap_peak_lod).to_csv("pap_peak_abs_clipped.csv", index=False)
pd.DataFrame(pap_p99_lod).to_csv("pap_p99_abs_clipped.csv", index=False)
pd.DataFrame(pap_rms1s_lod).to_csv("pap_max_1s_rms_clipped.csv", index=False)
pd.DataFrame(pap_spike_lod).to_csv("pap_spike_ratio_peak_over_p99.csv", index=False)
pd.DataFrame(pap_clipfrac_lod).to_csv("pap_fraction_samples_clipped.csv", index=False)

print("\nDone. Wrote PGV + PAP metric CSVs (peak, p99, 1s-RMS, spike ratio, clip fraction).")


Processing launch 1/140: falcon-9-full-thrust-ses-9  2016-03-04T23:35:00.000000Z -> 2016-03-05T01:06:00.000000Z
  Read 18 traces from event_outputs/20160304T233500Z/falcon-9-full-thrust-ses-9_raw.mseed
- removing instrument response
1R.BCHH.00.DD1 units=Pa
- removing instrument response
1R.BCHH.00.DD2 units=Pa
- removing instrument response
1R.BCHH.00.DD3 units=Pa
- removing instrument response
1R.BCHH.00.DHE units=m/s
- removing instrument response
1R.BCHH.00.DHN units=m/s
- removing instrument response
1R.BCHH.00.DHZ units=m/s
- removing instrument response
1R.FIRE.00.DD1 units=Pa
- removing instrument response
1R.FIRE.00.DD2 units=Pa
- removing instrument response
1R.FIRE.00.DD3 units=Pa
- removing instrument response
1R.FIRE.00.DHE units=m/s
- removing instrument response
1R.FIRE.00.DHN units=m/s
- removing instrument response
1R.FIRE.00.DHZ units=m/s
- removing instrument response
1R.TANK.00.DD1 units=Pa
- removing instrument response
1R.TANK.00.DD2 units=Pa
- removing instrument

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',


  Plotted seismic and infrasound traces for falcon-9-block-4-koreasat-5a

Processing launch 17/140: falcon-9-full-thrust-spx-crs-13  2017-12-15T15:36:09.000000Z -> 2017-12-15T15:36:09.000000Z
  Read 11 traces from event_outputs/20171215T153609Z/falcon-9-full-thrust-spx-crs-13_raw.mseed
- removing instrument response
Error removing response for 1R.BCHH1.00.DD1: No matching response information found.
- removing instrument response
1R.BCHH1.00.DD2 units=Pa
- removing instrument response
1R.BCHH1.00.DHE units=m/s
- removing instrument response
1R.BCHH1.00.DHN units=m/s
- removing instrument response
1R.BCHH1.00.DHZ units=m/s
- removing instrument response
1R.BCHH2.00.DD4 units=Pa
- removing instrument response
1R.BCHH2.00.DD5 units=Pa
- removing instrument response
1R.BCHH2.00.DD6 units=Pa
- removing instrument response
1R.BCHH2.00.DD7 units=Pa
- removing instrument response
1R.BCHH2.00.DD8 units=Pa
- removing instrument response
1R.BCHH2.00.DD9 units=Pa
  No usable samples after trimming

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_


Processing launch 21/140: falcon-heavy-demo-test-flight  2018-02-06T18:30:00.000000Z -> 2018-02-06T21:00:00.000000Z
  Read 23 traces from event_outputs/20180206T183000Z/falcon-heavy-demo-test-flight_raw.mseed
- removing instrument response
Error removing response for 1R.BCHH1.00.DD1: No matching response information found.
- removing instrument response
1R.BCHH1.00.DD2 units=Pa
- removing instrument response
1R.BCHH1.00.DHE units=m/s
- removing instrument response
1R.BCHH1.00.DHN units=m/s
- removing instrument response
1R.BCHH1.00.DHZ units=m/s
- removing instrument response
1R.BCHH2.00.DD4 units=Pa
- removing instrument response
1R.BCHH2.00.DD5 units=Pa
- removing instrument response
1R.BCHH2.00.DD6 units=Pa
- removing instrument response
1R.BCHH2.00.DD7 units=Pa
- removing instrument response
1R.BCHH2.00.DD8 units=Pa
- removing instrument response
1R.BCHH2.00.DD9 units=Pa
- removing instrument response
1R.DVEL1.00.DD1 units=Pa
- removing instrument response
1R.DVEL1.00.DD2 units=Pa

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',


  Plotted seismic and infrasound traces for falcon-heavy-demo-test-flight

Processing launch 22/140: atlas-v-541-goes-s  2018-03-01T22:02:00.000000Z -> 2018-03-02T00:02:00.000000Z
  Read 11 traces from event_outputs/20180301T220200Z/atlas-v-541-goes-s_raw.mseed
- removing instrument response
Error removing response for 1R.BCHH1.00.DD1: No matching response information found.
- removing instrument response
1R.BCHH1.00.DD2 units=Pa
- removing instrument response
1R.BCHH1.00.DHE units=m/s
- removing instrument response
1R.BCHH1.00.DHN units=m/s
- removing instrument response
1R.BCHH1.00.DHZ units=m/s
- removing instrument response
1R.BCHH2.00.DD4 units=Pa
- removing instrument response
1R.BCHH2.00.DD5 units=Pa
- removing instrument response
1R.BCHH2.00.DD6 units=Pa
- removing instrument response
1R.BCHH2.00.DD7 units=Pa
- removing instrument response
1R.BCHH2.00.DD8 units=Pa
- removing instrument response
1R.BCHH2.00.DD9 units=Pa
  Plotted seismic and infrasound traces for atlas-v-541-goe

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small')
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/waveform.py:815: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), fontsize='small',
/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/imaging/util.py:266: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_

  Plotted seismic and infrasound traces for falcon-9-block-5-nilesat-301

Processing launch 112/140: astra-rocket-3-tropics-1  2022-06-12T16:00:00.000000Z -> 2022-06-12T18:00:00.000000Z
  Read 19 traces from event_outputs/20220612T160000Z/astra-rocket-3-tropics-1_raw.mseed
- removing instrument response
1R.BCHH2.00.HD4 units=Pa
- removing instrument response
1R.BCHH2.00.HD5 units=Pa
- removing instrument response
1R.BCHH2.00.HD6 units=Pa
- removing instrument response
1R.BCHH2.00.HD7 units=Pa
- removing instrument response
1R.BCHH2.00.HD8 units=Pa
- removing instrument response
1R.BCHH2.00.HD9 units=Pa
- removing instrument response
1R.BCHH4.00.HD2 units=Pa
- removing instrument response
Error removing response for 1R.BCHH4.00.HDF: No matching response information found.
- removing instrument response
1R.BCHH4.00.HHE units=m/s
- removing instrument response
1R.BCHH4.00.HHN units=m/s
- removing instrument response
1R.BCHH4.00.HHZ units=m/s
- removing instrument response
1R.S39A1.00.HDF 

# GOT HERE

In [10]:
raise NotImplementedError('stop here')

NotImplementedError: stop here

In [ ]:
from flovopy.core.miniseed_io import read_mseed 
for i, row in event_df.iterrows():
    basepath = row['basepath']
    slug = row['slug']
    t0 = row['t0']
    t1 = row['t1']
    st = read_mseed(row['mseed_path'])
    try:
        # NOTE: your preprocess_stream signature might accept freq=0.1 or freq=[0.1]
        # If it expects list, change to freq=[args.hp]
        st_processed = preprocess_stream(
            st,
            freq=hp,
            filter_type="highpass",
            inv=inv,
            verbose=verbose,
        )
        log("  preprocess_stream succeeded", verbose, 2)
    except Exception as e:
        log(f"  preprocess_stream FAILED: {type(e).__name__}: {e}", verbose, 1)
        index_rows.append(dict(
            slug=slug,
            name=getattr(r, "name", ""),
            window_start=str(r.window_start),
            window_end=str(r.window_end),
            status="preprocess_fail",
            error=f"{type(e).__name__}: {e}",
            basepath=basepath,
        ))
        continue


    # --- Ensure EnhancedStream ---
    try:
        est = st_processed if isinstance(st_processed, EnhancedStream) else EnhancedStream(st_processed)
    except Exception as e:
        bad_row_info("EnhancedStream FAILED:", r, index_rows, slug, basepath, e, verbose_level=1)
        log(f"  EnhancedStream FAILED: {type(e).__name__}: {e}", args.verbose, 1)
        continue

    # --- Trim to EXACT launch window to avoid taper bias ---
    try:
        est_win = est.copy().trim(t0, t1, pad=False)
    except Exception as e:
        log(f"  WARN trim failed; using untrimmed stream: {e}", args.verbose, 1)
        est_win = est

    # --- Compute metrics on the TRIMMED window ---
    try:
        est_win.ampengfft()
        log("  ampengfft succeeded (trimmed)", args.verbose, 2)
    except Exception as e:
        log(f"  ampengfft FAILED: {type(e).__name__}: {e}", args.verbose, 1)
        index_rows.append(dict(
            slug=slug,
            name=getattr(r, "name", ""),
            window_start=str(r.window_start),
            window_end=str(r.window_end),
            status="ampengfft_fail",
            error=f"{type(e).__name__}: {e}",
            basepath=basepath,
        ))
        continue

    # --- Save bundle (waveforms + flattened metrics CSVs) ---
    try:
        written = save_enhancedstream_bundle(
            est_win,
            basepath,
            save_pickle=args.save_pickle,
            verbose=args.verbose,
        )
        status = "ok"
        error = ""
    except Exception as e:
        written = {}
        status = "save_fail"
        error = f"{type(e).__name__}: {e}"
        log(f"  SAVE FAILED: {error}", args.verbose, 1)

    # --- Add one summary row per launch ---
    index_rows.append(dict(
        slug=slug,
        name=getattr(r, "name", ""),
        launch_designator=getattr(r, "launch_designator", ""),
        slc=getattr(r, "SLC", ""),
        launch_success_flag=getattr(r, "success", ""),
        window_start=str(r.window_start),
        window_end=str(r.window_end),
        pad_s=float(args.pad_s),
        hp_hz=float(args.hp),
        n_traces=int(len(est_win)) if est_win else 0,
        status=status,
        error=error,
        basepath=basepath,
        mseed_path=written.get("mseed", ""),
        trace_csv_path=written.get("trace_csv", ""),
        station_csv_path=written.get("station_csv", ""),
        pickle_path=written.get("pickle", ""),
    ))

    # Cut length for plots - placeholder for detection or real times
    est_win.trim(endtime=t0+900)

    # Make any other plots - and add spectrograms too
    temp_st = est_win.select(component="Z")
    temp_st.select(component="Z").plot(outfile=basepath + "_Z_traces.png", size=(1200, 800))
    spobj_Z = icewebSpectrogram(temp_st)
    spobj_Z.plot(outfile=basepath + "_Z_spectrograms.png")#, size=(1200, 800))

    # Find unique stations in this launch
    stations = set(tr.stats.station for tr in est_win)
    for sta in stations:
        temp_st=est_win.select(station=sta)
        temp_st.plot(outfile=basepath + f"_{sta}_traces.png", size=(1200, 800))
        spobj_sta = icewebSpectrogram(temp_st.select(station=sta))
        spobj_sta.plot(outfile=basepath + f"_{sta}_spectrograms.png")#, size=(1200, 800))

    # plot the first infrasound channel at each station, for all stations in one figure
    infrasound_stream = EnhancedStream()
    for sta in stations:
        p_trs = est_win.select(station=sta).select(channel="*D*")
        if p_trs:
            p_tr = p_trs[0]
            infrasound_stream.append(p_tr)
    if len(infrasound_stream) > 0:
        infrasound_stream.plot(outfile=basepath + f"_P_traces.png", size=(1200, 800))
        spobj_P = icewebSpectrogram(infrasound_stream)
        spobj_P.plot(outfile=basepath + f"_P_spectrograms.png")#, size=(1200, 800))




In [ ]:

# Write index CSV
out_index = pd.DataFrame(index_rows)
out_index.to_csv(args.index_out, index=False)
print(f"\nWrote index: {args.index_out} ({len(out_index):,} launches)")



In [ ]:
        # --- Ensure EnhancedStream ---
        try:
            est = st_processed if isinstance(st_processed, EnhancedStream) else EnhancedStream(st_processed)
        except Exception as e:
            log(f"  EnhancedStream FAILED: {type(e).__name__}: {e}", args.verbose, 1)
            index_rows.append(dict(
                slug=slug,
                name=getattr(r, "name", ""),
                window_start=str(r.window_start),
                window_end=str(r.window_end),
                status="enhancedstream_fail",
                error=f"{type(e).__name__}: {e}",
                basepath=basepath,
            ))
            continue

        # --- Trim to EXACT launch window to avoid taper bias ---
        try:
            est_win = est.copy().trim(t0, t1, pad=False)
        except Exception as e:
            log(f"  WARN trim failed; using untrimmed stream: {e}", args.verbose, 1)
            est_win = est

        # --- Compute metrics on the TRIMMED window ---
        try:
            est_win.ampengfft()
            log("  ampengfft succeeded (trimmed)", args.verbose, 2)
        except Exception as e:
            log(f"  ampengfft FAILED: {type(e).__name__}: {e}", args.verbose, 1)
            index_rows.append(dict(
                slug=slug,
                name=getattr(r, "name", ""),
                window_start=str(r.window_start),
                window_end=str(r.window_end),
                status="ampengfft_fail",
                error=f"{type(e).__name__}: {e}",
                basepath=basepath,
            ))
            continue

        # --- Save bundle (waveforms + flattened metrics CSVs) ---
        try:
            written = save_enhancedstream_bundle(
                est_win,
                basepath,
                save_pickle=args.save_pickle,
                verbose=args.verbose,
            )
            status = "ok"
            error = ""
        except Exception as e:
            written = {}
            status = "save_fail"
            error = f"{type(e).__name__}: {e}"
            log(f"  SAVE FAILED: {error}", args.verbose, 1)

        # --- Add one summary row per launch ---
        index_rows.append(dict(
            slug=slug,
            name=getattr(r, "name", ""),
            launch_designator=getattr(r, "launch_designator", ""),
            slc=getattr(r, "SLC", ""),
            launch_success_flag=getattr(r, "success", ""),
            window_start=str(r.window_start),
            window_end=str(r.window_end),
            pad_s=float(args.pad_s),
            hp_hz=float(args.hp),
            n_traces=int(len(est_win)) if est_win else 0,
            status=status,
            error=error,
            basepath=basepath,
            mseed_path=written.get("mseed", ""),
            trace_csv_path=written.get("trace_csv", ""),
            station_csv_path=written.get("station_csv", ""),
            pickle_path=written.get("pickle", ""),
        ))

        # Cut length for plots - placeholder for detection or real times
        est_win.trim(endtime=t0+900)

        # Make any other plots - and add spectrograms too
        temp_st = est_win.select(component="Z")
        temp_st.select(component="Z").plot(outfile=basepath + "_Z_traces.png", size=(1200, 800))
        spobj_Z = icewebSpectrogram(temp_st)
        spobj_Z.plot(outfile=basepath + "_Z_spectrograms.png")#, size=(1200, 800))

        # Find unique stations in this launch
        stations = set(tr.stats.station for tr in est_win)
        for sta in stations:
            temp_st=est_win.select(station=sta)
            temp_st.plot(outfile=basepath + f"_{sta}_traces.png", size=(1200, 800))
            spobj_sta = icewebSpectrogram(temp_st.select(station=sta))
            spobj_sta.plot(outfile=basepath + f"_{sta}_spectrograms.png")#, size=(1200, 800))

        # plot the first infrasound channel at each station, for all stations in one figure
        infrasound_stream = EnhancedStream()
        for sta in stations:
            p_trs = est_win.select(station=sta).select(channel="*D*")
            if p_trs:
                p_tr = p_trs[0]
                infrasound_stream.append(p_tr)
        if len(infrasound_stream) > 0:
            infrasound_stream.plot(outfile=basepath + f"_P_traces.png", size=(1200, 800))
            spobj_P = icewebSpectrogram(infrasound_stream)
            spobj_P.plot(outfile=basepath + f"_P_spectrograms.png")#, size=(1200, 800))



    # Write index CSV
    out_index = pd.DataFrame(index_rows)
    out_index.to_csv(args.index_out, index=False)
    print(f"\nWrote index: {args.index_out} ({len(out_index):,} launches)")